In [53]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'


class CnnMnist(nn.Module):
    def __init__(self, in_c, n_classes, img_w, img_h):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels=in_c,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        # Calcula automaticamente o tamanho da entrada do Linear
        dummy_input = torch.zeros(1, in_c, img_h, img_w)
        dummy_output = self.conv(dummy_input)

        in_features = dummy_output.flatten(1).shape[1]

        self.ffn = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, n_classes)
        )


    def forward(self, x):
        x = self.conv(x)

        x = torch.flatten(x, 1)

        x = self.ffn(x)

        return x


model = CnnMnist(
    in_c=1,
    n_classes=10,
    img_w=28,
    img_h=28
).to(device)


print("Model ready!")
print(f"Device: {next(model.parameters()).device}")


Model ready!
Device: cuda:0


In [54]:
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root='./data',      
    train=True,         
    download=True,      
    transform=transform 
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,        
    download=True,
    transform=transform
)

In [55]:
train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=20, 
    shuffle=True 
)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=20, 
    shuffle=False
)


In [56]:
import torch.nn as nn
import torch.optim as optim


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [57]:
epochs = 1

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # enumerate() nos dá o número do batch (i) e a dupla (img, label)
    for i, (img, label) in enumerate(train_loader):
        img = img.to(device)
        label = label.to(device)

        # 1. Forward Pass
        logits = model(img)
        loss = criterion(logits, label)

        # 2. Backward Pass & Atualização dos Pesos
        optimizer.zero_grad()  # Zera antes do backward (boa prática)
        loss.backward()
        optimizer.step()

        # Acumula a perda para métricas
        running_loss += loss.item()

        # Imprime o progresso a cada 100 batches (para não poluir o terminal)
        if i % 100 == 0:
            print(f'Época [{epoch+1}/{epochs}] | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}')


    # Loss média da época
    epoch_loss = running_loss / len(train_loader)
    print(f'==> Fim da Época {epoch+1} | Loss Média: {epoch_loss:.4f}\n')

Época [1/1] | Batch 0/3000 | Loss: 2.5182
Época [1/1] | Batch 100/3000 | Loss: 0.3603
Época [1/1] | Batch 200/3000 | Loss: 0.1570
Época [1/1] | Batch 300/3000 | Loss: 0.0824
Época [1/1] | Batch 400/3000 | Loss: 0.1686
Época [1/1] | Batch 500/3000 | Loss: 0.0757
Época [1/1] | Batch 600/3000 | Loss: 0.0140
Época [1/1] | Batch 700/3000 | Loss: 0.3548
Época [1/1] | Batch 800/3000 | Loss: 0.0183
Época [1/1] | Batch 900/3000 | Loss: 0.0227
Época [1/1] | Batch 1000/3000 | Loss: 0.0759
Época [1/1] | Batch 1100/3000 | Loss: 0.1359
Época [1/1] | Batch 1200/3000 | Loss: 0.0149
Época [1/1] | Batch 1300/3000 | Loss: 0.0187
Época [1/1] | Batch 1400/3000 | Loss: 0.0172
Época [1/1] | Batch 1500/3000 | Loss: 0.1062
Época [1/1] | Batch 1600/3000 | Loss: 0.0171
Época [1/1] | Batch 1700/3000 | Loss: 0.0483
Época [1/1] | Batch 1800/3000 | Loss: 0.0209
Época [1/1] | Batch 1900/3000 | Loss: 0.0319
Época [1/1] | Batch 2000/3000 | Loss: 0.0154
Época [1/1] | Batch 2100/3000 | Loss: 0.1113
Época [1/1] | Batch 22

In [58]:
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for img, label in test_loader:
        img = img.to(device)
        label = label.to(device)

        logits = model(img)
        loss = criterion(logits, label)

        test_loss += loss.item()

            # Calcula acurácia
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == label).sum().item()
        total += label.size(0)

    test_loss /= len(test_loader)
    accuracy = 100 * correct / total

print(f'==> Teste | Loss: {test_loss:.4f} | Acurácia: {accuracy:.2f}%\n')

==> Teste | Loss: 0.0367 | Acurácia: 98.90%



In [60]:
if test_loss > 0.05:
    print('Accuracy necessária não atingida!')
else:
    print(f'Accuracy atingida! salvando modelo')
    torch.save(model.state_dict(), "mnist_cnn.pth")

Accuracy atingida! salvando modelo
